# Transform Circuits Data

1. Read bronze _circuits_ table
2. Keep only the columns required for analytics (Drop _url_ column).
3. Standardise column names using snake_case (_circuitId_ -> _circuit_id_ for example)
4. Rename columns to make them more meaningful (_lat_ -> latitude for example)
5. Filter out rows where _circuit_id_ is null (Business Key validation)
6. Remove duplicate records
7. Transform values of columns _circuit_name_ and _locality_ to Title Case
8. Write the transformed data to silver circuits table.

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

### Step 1 - Read bronze _circuits_ table

In [0]:
# Methode 1 that allows options since it's an API call:
# circuits_df = spark.read.table(bronze_table)

In [0]:
# Method 2 using a method with no additionnal options
circuits_df = spark.table(bronze_table)

In [0]:
display(circuits_df)

**2. Keep only the columns required for analytics (Drop _url_ column).**

In [0]:
# circuits_selected_df = circuits_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "long",
#     "locality",
#     "country",
#     "ingestion_timestamp",
#     "source_file"
# )

_AUTRE METHODE POUR KEEP LES BONNES COLONNES : EST PLUS FLEXIBLE CAR PERMET D'AJOUTER DES ALIAS DIRECTEMENT PAR EXEMPLE OU ENCORE DE CHANGER LES VALUERS D'UNE COLONNE!_

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

**3. Standardise column names using snake_case (_circuitId_ -> _circuit_id_ for example) &
4. Rename columns to make them more meaningful (_lat_ -> latitude for example)**

In [0]:
# ANCIENNE VERSION!!
# circuits_renamed_df = (
#     circuits_selected_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("circuitName", "circuit_name")
#     .withColumnRenamed("lat", "latitude")
#     .withColumnRenamed("long","longitude")                       
# )

In [0]:
circuits_renamed_df = (
    circuits_selected_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "circuitName": "circuit_name",
        "lat": "latitude",
        "long": "longitude" 
    })                     
)

In [0]:
display(circuits_renamed_df)

**5. Filter out rows where _circuit_id_ is null (Business Key validation)**

In [0]:
# METHOD 1 SQL:
# circuits_valid_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )

In [0]:
# METHOD 2 PYSPARK:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
display(circuits_valid_df)

**6. Remove duplicate records**

In [0]:
# circuits_distinct_df = circuits_valid_df.distinct()

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

**7. Transform values of columns _circuit_name_ and _locality_ to Title Case**

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
        .withColumn("locality", F.initcap(F.col("locality")))
)

In [0]:
display(circuits_final_df)

**8. Write the transformed data to silver circuits table.**

In [0]:
(
    circuits_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))